# 02_feature_engineering.ipynb

This notebook loads the raw dataset from Amazon S3, performs exploratory data profiling, applies feature engineering transformations, creates and populates a SageMaker Feature Store feature group, and prepares the processed dataset for downstream model training.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [ ]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and project modules.

In [ ]:
import time
from pathlib import Path

import boto3
import pandas as pd

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    FEATURE_GROUP_NAME,
    FEATURE_STORE_S3_URI,
    PROCESSED_DATA_KEY,
    RAW_DATA_KEY,
    TARGET_COLUMN,
)
from src.preprocess_simplified import preprocess_dataframe

## Initialize AWS Clients

Initialize clients for Amazon S3, SageMaker, and SageMaker Feature Store.

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)

sm = boto3.client(
    "sagemaker",
    region_name=AWS_REGION,
)

featurestore_runtime = boto3.client(
    "sagemaker-featurestore-runtime",
    region_name=AWS_REGION,
)

sts = boto3.client(
    "sts",
    region_name=AWS_REGION,
)

## Load Raw Dataset from Amazon S3

The raw dataset created during the ingestion step is loaded from Amazon S3 for profiling and feature engineering.

In [ ]:
obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=RAW_DATA_KEY
)

df = pd.read_csv(obj["Body"])

## Dataset Preview

Inspect the first rows of the dataset and verify that the raw data was loaded correctly.

In [ ]:
df.head()

## Data Quality Validation

Perform a quick validation to inspect missing values and confirm the dataset structure.

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

missing_values = df.isnull().sum()

print(f"Total missing values: {missing_values.sum()}")

## Data Profiling Summary

The data profiling step in AWS SageMaker Data Wrangler identified the following observations:

- No missing values were detected in the dataset.
- The target distribution is moderately imbalanced, with approximately 70% `good` and 30% `bad`.
- Numerical feature distributions show meaningful variation across the dataset.
- Several moderate correlations were identified between selected features.
- The dataset contains a mixture of numerical and categorical features.

## Feature Engineering

No missing values are detected during data profiling, and therefore no imputation is required.

Categorical features are transformed using one-hot encoding in SageMaker Data Wrangler to prepare the dataset for model training with XGBoost.

The transformation workflow is implemented in Data Wrangler and exported as Python code for reuse within the project and future pipeline automation.

In [ ]:
processed_df = preprocess_dataframe(df)

## Preview Processed Dataset

Inspect the first rows of the processed dataset.

In [ ]:
processed_df.head()

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,class,checking_status_0le_Xlt_200,checking_status_lt_0,...,job_skilled,job_unemp_unskilled_non_res,job_unskilled_resident,own_telephone_none,own_telephone_yes,foreign_worker_no,foreign_worker_yes,other_payment_plans_bank,other_payment_plans_none,other_payment_plans_stores
0,6,1169,4,4,67,2,1,1,0,1,...,1,0,0,0,1,0,1,0,1,0
1,48,5951,2,2,22,1,1,0,1,0,...,1,0,0,1,0,0,1,0,1,0
2,12,2096,2,3,49,1,2,1,0,0,...,0,0,1,1,0,0,1,0,1,0
3,42,7882,2,4,45,1,2,1,0,1,...,1,0,0,1,0,0,1,0,1,0
4,24,4870,3,4,53,2,2,0,0,1,...,1,0,0,1,0,0,1,0,1,0


## Prepare Feature Store Dataset

Prepare the processed dataset for SageMaker Feature Store by adding metadata columns and removing the target variable.

In [ ]:
# Start from the processed dataframe
feature_df = processed_df.drop(columns=[TARGET_COLUMN]).copy()

# Feature Store metadata columns
feature_df["record_id"] = feature_df.index.astype(str)
feature_df["event_time"] = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

feature_df.head()

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,checking_status_0le_Xlt_200,checking_status_lt_0,checking_status_ge_200,...,job_unskilled_resident,own_telephone_none,own_telephone_yes,foreign_worker_no,foreign_worker_yes,other_payment_plans_bank,other_payment_plans_none,other_payment_plans_stores,record_id,event_time
0,6,1169,4,4,67,2,1,0,1,0,...,0,0,1,0,1,0,1,0,0,2026-06-19T13:09:51Z
1,48,5951,2,2,22,1,1,1,0,0,...,0,1,0,0,1,0,1,0,1,2026-06-19T13:09:51Z
2,12,2096,2,3,49,1,2,0,0,0,...,1,1,0,0,1,0,1,0,2,2026-06-19T13:09:51Z
3,42,7882,2,4,45,1,2,0,1,0,...,0,1,0,0,1,0,1,0,3,2026-06-19T13:09:51Z
4,24,4870,3,4,53,2,2,0,1,0,...,0,1,0,0,1,0,1,0,4,2026-06-19T13:09:51Z


## Retrieve SageMaker Execution Role

Retrieve the execution role associated with the SageMaker domain. This role is required to create and manage SageMaker Feature Store resources.

In [ ]:
domain = sm.list_domains()["Domains"][0]

domain_details = sm.describe_domain(
    DomainId=domain["DomainId"]
)

ROLE_ARN = domain_details["DefaultUserSettings"]["ExecutionRole"]
print(ROLE_ARN)

arn:aws:iam::591874026136:role/service-role/AmazonSageMaker-ExecutionRole-20260615T111026


## Create Feature Definitions

Create the Feature Store schema by converting dataframe column types into SageMaker Feature Store feature definitions.

In [ ]:
feature_definitions = []

for column, dtype in feature_df.dtypes.items():

    if str(dtype) == "object":
        feature_type = "String"

    elif str(dtype) in ["int64", "int32", "bool"]:
        feature_type = "Integral"

    elif str(dtype) in ["float64", "float32"]:
        feature_type = "Fractional"

    else:
        raise ValueError(
            f"Unsupported dtype '{dtype}' for column '{column}'"
        )

    feature_definitions.append(
        {
            "FeatureName": column,
            "FeatureType": feature_type,
        }
    )
feature_definitions

[{'FeatureName': 'duration', 'FeatureType': 'Integral'},
 {'FeatureName': 'credit_amount', 'FeatureType': 'Integral'},
 {'FeatureName': 'installment_commitment', 'FeatureType': 'Integral'},
 {'FeatureName': 'residence_since', 'FeatureType': 'Integral'},
 {'FeatureName': 'age', 'FeatureType': 'Integral'},
 {'FeatureName': 'existing_credits', 'FeatureType': 'Integral'},
 {'FeatureName': 'num_dependents', 'FeatureType': 'Integral'},
 {'FeatureName': 'checking_status_0le_Xlt_200', 'FeatureType': 'Integral'},
 {'FeatureName': 'checking_status_lt_0', 'FeatureType': 'Integral'},
 {'FeatureName': 'checking_status_ge_200', 'FeatureType': 'Integral'},
 {'FeatureName': 'checking_status_no_checking', 'FeatureType': 'Integral'},
 {'FeatureName': 'credit_history_all_paid', 'FeatureType': 'Integral'},
 {'FeatureName': 'credit_history_critical_other_existing_credit',
  'FeatureType': 'Integral'},
 {'FeatureName': 'credit_history_delayed_previously',
  'FeatureType': 'Integral'},
 {'FeatureName': 'cred

## Create Feature Group

Create the SageMaker Feature Store feature group using the generated feature definitions.

In [ ]:
try:
    sm.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )

    print(f"Feature Group '{FEATURE_GROUP_NAME}' already exists.")

except sm.exceptions.ResourceNotFound:
    print(f"Creating Feature Group '{FEATURE_GROUP_NAME}'...")

    sm.create_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierFeatureName="record_id",
        EventTimeFeatureName="event_time",
        FeatureDefinitions=feature_definitions,
        OnlineStoreConfig={
            "EnableOnlineStore": True
        },
        OfflineStoreConfig={
            "S3StorageConfig": {
                "S3Uri": FEATURE_STORE_S3_URI
            }
        },
        RoleArn=ROLE_ARN,
    )

Feature Group 'credit-risk-features' already exists.


## Wait for Feature Group Creation

Wait until the feature group status changes to `Created` before ingesting any records.

In [ ]:
while True:
    status = sm.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )["FeatureGroupStatus"]

    print(f"Status: {status}")

    if status == "Created":
        print("Feature Group is ready.")
        break

    if status == "CreateFailed":
        raise RuntimeError("Feature Group creation failed.")

    time.sleep(10)

Status: Created
Feature Group is ready.


## Ingest Features into Feature Store

Write the processed feature records to SageMaker Feature Store.

In [ ]:
for _, row in feature_df.iterrows():

    record = []

    for col in feature_df.columns:
        record.append(
            {
                "FeatureName": col,
                "ValueAsString": str(row[col])
            }
        )

    featurestore_runtime.put_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        Record=record
    )

## Verify Feature Store Records

Read back a sample record to confirm that ingestion worked as expected.

In [ ]:
featurestore_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    RecordIdentifierValueAsString="0"
)

{'ResponseMetadata': {'RequestId': 'cbb4cfc0-2d7c-4b1e-9652-a262209325de',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'cbb4cfc0-2d7c-4b1e-9652-a262209325de',
   'content-type': 'application/json',
   'content-length': '5431',
   'date': 'Fri, 19 Jun 2026 13:10:23 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'duration', 'ValueAsString': '6'},
  {'FeatureName': 'credit_amount', 'ValueAsString': '1169'},
  {'FeatureName': 'installment_commitment', 'ValueAsString': '4'},
  {'FeatureName': 'residence_since', 'ValueAsString': '4'},
  {'FeatureName': 'age', 'ValueAsString': '67'},
  {'FeatureName': 'existing_credits', 'ValueAsString': '2'},
  {'FeatureName': 'num_dependents', 'ValueAsString': '1'},
  {'FeatureName': 'checking_status_0le_Xlt_200', 'ValueAsString': '0'},
  {'FeatureName': 'checking_status_lt_0', 'ValueAsString': '1'},
  {'FeatureName': 'checking_status_ge_200', 'ValueAsString': '0'},
  {'FeatureName': 'checking_status_no_checking', 'ValueAsStrin

## Upload to S3

Store a local copy of the processed dataset and upload it to the processed data layer in Amazon S3.

In [ ]:
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
local_file_name = "data/processed/german_credit.csv"

processed_df.to_csv(
    local_file_name,
    index=False
)

In [ ]:
s3.upload_file(
    local_file_name,
    BUCKET_NAME,
    PROCESSED_DATA_KEY
)

print(f"Uploaded to s3://{BUCKET_NAME}/{PROCESSED_DATA_KEY}")

Uploaded to s3://aws-sagemaker-showcase/processed/german_credit_processed.csv


In [ ]:
response = s3.head_object(
    Bucket=BUCKET_NAME,
    Key=PROCESSED_DATA_KEY
)

print("Upload successful")

Upload successful


## Result

The processed dataset is now stored in Amazon S3.